<a href="https://colab.research.google.com/github/Andre-lab/pde-workshop-ai-protein-design/blob/main/Task_1_New_ThreadSequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release

Looking in links: https://west.rosettacommons.org/pyrosetta/quarterly/release
     ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 GB 29.0 MB/s eta 0:00:56
ERROR: Operation cancelled by user


In [ ]:
#!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=1)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *

ModuleNotFoundError: No module named 'pyrosettacolabsetup'

In [ ]:
import os
import shutil

import pyrosetta
from pyrosetta import rosetta

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

In [ ]:
pyrosetta.init("-mute all")
pdb_path = None
original_pdb_name = None

upload_pdb_button = widgets.Button(
    description="Upload PDB",
    button_style="info",
    icon="upload"
)

upload_pdb_output = widgets.Output()


def upload_pdb(_):
    global pdb_path, original_pdb_name

    with upload_pdb_output:
        clear_output()

        uploaded = files.upload()

        pdbs = [
            name for name in uploaded.keys()
            if name.lower().endswith(".pdb")
        ]

        if not pdbs:
            print("No PDB file was uploaded.")
            return

        original_pdb_name = pdbs[0]

        # Use a simple internal filename
        pdb_path = "/content/input_structure.pdb"

        shutil.copy2(
            original_pdb_name,
            pdb_path
        )

        print("Uploaded:")
        print(f"  {original_pdb_name}")
        print()
        print("Internal path:")
        print(f"  {pdb_path}")


upload_pdb_button.on_click(upload_pdb)

display(
    upload_pdb_button,
    upload_pdb_output
)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

Button(button_style='info', description='Upload PDB', icon='upload', style=ButtonStyle())

Output()

In [ ]:
sequence_widget = widgets.Textarea(
    value="",
    placeholder=(
        "Paste the target protein sequence here.\n"
        "Example:\n"
        "MKTIIALSYIFCLVFADYKDDDDK..."
    ),
    description="Sequence:",
    layout=widgets.Layout(
        width="700px",
        height="180px"
    )
)

display(sequence_widget)

Textarea(value='', description='Sequence:', layout=Layout(height='180px', width='700px'), placeholder='Paste t…

In [ ]:
start_residue_widget = widgets.IntText(
    value=1,
    description="Start residue:",
    layout=widgets.Layout(width="250px")
)

display(start_residue_widget)

IntText(value=1, description='Start residue:', layout=Layout(width='250px'))

In [ ]:
VALID_AAS = set(
    "ACDEFGHIKLMNPQRSTVWY"
)


def clean_sequence(sequence_text):

    lines = []

    for line in sequence_text.splitlines():

        line = line.strip()

        if not line:
            continue

        # Ignore FASTA header
        if line.startswith(">"):
            continue

        lines.append(line)

    sequence = "".join(lines)
    sequence = sequence.replace(" ", "")
    sequence = sequence.upper()

    return sequence


def validate_sequence(sequence):

    if not sequence:
        raise ValueError(
            "No sequence was provided."
        )

    invalid = sorted(
        set(sequence) - VALID_AAS
    )

    if invalid:
        raise ValueError(
            "Invalid amino-acid characters: "
            + ", ".join(invalid)
        )

    return sequence

In [ ]:
def load_pose():

    if pdb_path is None:
        raise ValueError(
            "Please upload a PDB first."
        )

    if not os.path.exists(pdb_path):
        raise FileNotFoundError(
            pdb_path
        )

    pose = pyrosetta.pose_from_pdb(
        pdb_path
    )

    return pose

In [ ]:
pose = load_pose()

print(
    "PDB:",
    original_pdb_name
)

print(
    "Pose residues:",
    pose.total_residue()
)

print(
    "Current sequence length:",
    len(pose.sequence())
)

print(
    "Current sequence:"
)

print(
    pose.sequence()
)

PDB: run_M0179_1q3s_cond0_99-atomized-bb-True.pdb
Pose residues: 182
Current sequence length: 182
Current sequence:
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTAAAAAAAAAZZ


In [ ]:
def thread_sequence(
    pose,
    sequence,
    start_residue
):

    # Check that the sequence fits on the pose
    end_residue = (
        start_residue
        + len(sequence)
        - 1
    )

    if start_residue < 1:
        raise ValueError(
            "Start residue must be >= 1."
        )

    if end_residue > pose.total_residue():
        raise ValueError(
            f"Sequence is too long for the "
            f"selected starting position.\n"
            f"Sequence length: {len(sequence)}\n"
            f"Start residue: {start_residue}\n"
            f"End residue: {end_residue}\n"
            f"Pose length: {pose.total_residue()}"
        )

    # Make a copy so the original pose remains untouched
    threaded_pose = pose.clone()

    # SimpleThreadingMover
    threader = (
        rosetta.protocols.simple_moves
        .SimpleThreadingMover(
            sequence,
            start_residue
        )
    )

    # Apply threading
    threader.apply(threaded_pose)

    return threaded_pose

In [ ]:
from pyrosetta import rosetta


def thread_sequence(
    pose,
    sequence,
    start_residue
):
    """
    Thread `sequence` onto an existing Pose backbone,
    starting at the given Pose residue.
    """

    if start_residue < 1:
        raise ValueError(
            "start_residue must be >= 1."
        )

    end_residue = (
        start_residue
        + len(sequence)
        - 1
    )

    if end_residue > pose.total_residue():
        raise ValueError(
            f"Sequence is too long for the selected "
            f"starting position.\n"
            f"Sequence length: {len(sequence)}\n"
            f"Start residue: {start_residue}\n"
            f"End residue: {end_residue}\n"
            f"Pose length: {pose.total_residue()}"
        )

    # Work on a copy
    threaded_pose = pose.clone()

    # Apply SimpleThreadingMover
    threader = (
        rosetta.protocols.simple_moves
        .SimpleThreadingMover(
            sequence,
            start_residue
        )
    )

    threader.apply(threaded_pose)

    return threaded_pose

In [ ]:
run_button = widgets.Button(
    description="Thread Sequence",
    button_style="success",
    icon="play"
)

run_output = widgets.Output()

threaded_pose = None


def run_threading(_):

    global threaded_pose

    with run_output:

        clear_output()

        try:

            # --------------------------------------------
            # Clean sequence
            # --------------------------------------------

            sequence = clean_sequence(
                sequence_widget.value
            )

            sequence = validate_sequence(
                sequence
            )

            # --------------------------------------------
            # Load PDB
            # --------------------------------------------

            pose = load_pose()

            # --------------------------------------------
            # Start residue
            # --------------------------------------------

            start_residue = (
                start_residue_widget.value
            )

            print(
                "Input PDB:",
                original_pdb_name
            )

            print(
                "Pose length:",
                pose.total_residue()
            )

            print(
                "Target sequence length:",
                len(sequence)
            )

            print(
                "Start residue:",
                start_residue
            )

            print()

            # --------------------------------------------
            # Thread
            # --------------------------------------------

            print(
                "Running SimpleThreadingMover..."
            )

            threaded_pose = thread_sequence(
                pose,
                sequence,
                start_residue
            )

            print(
                "Threading complete."
            )

            # --------------------------------------------
            # Verify result
            # --------------------------------------------

            end_residue = (
                start_residue
                + len(sequence)
                - 1
            )

            resulting_segment = (
                threaded_pose
                .sequence()[
                    start_residue - 1:
                    end_residue
                ]
            )

            print()
            print(
                "Expected sequence:"
            )
            print(sequence)

            print()
            print(
                "Threaded sequence:"
            )
            print(resulting_segment)

            if resulting_segment == sequence:
                print()
                print(
                    "Sequence verification: PASSED"
                )
            else:
                print()
                print(
                    "Sequence verification: FAILED"
                )

        except Exception as e:

            print(
                "ERROR:",
                type(e).__name__,
                e
            )


run_button.on_click(
    run_threading
)

display(
    run_button,
    run_output
)

Button(button_style='success', description='Thread Sequence', icon='play', style=ButtonStyle())

Output()

In [ ]:
output_name_widget = widgets.Text(
    value="threaded_model.pdb",
    description="Output filename:",
    layout=widgets.Layout(width="400px")
)

display(output_name_widget)

Text(value='threaded_model.pdb', description='Output filename:', layout=Layout(width='400px'))

In [ ]:
save_button = widgets.Button(
    description="Save Threaded PDB",
    button_style="warning",
    icon="save"
)

save_output = widgets.Output()


def save_threaded_pdb(_):

    with save_output:

        clear_output()

        try:

            if threaded_pose is None:
                raise ValueError(
                    "Run the threading calculation first."
                )

            filename = (
                output_name_widget.value.strip()
            )

            if not filename:
                filename = "threaded_model.pdb"

            if not filename.lower().endswith(
                ".pdb"
            ):
                filename += ".pdb"

            output_path = os.path.join(
                "/content",
                filename
            )

            threaded_pose.dump_pdb(
                output_path
            )

            print(
                "Saved:",
                output_path
            )

        except Exception as e:

            print(
                "ERROR:",
                type(e).__name__,
                e
            )


save_button.on_click(
    save_threaded_pdb
)

display(
    save_button,
    save_output
)

Button(button_style='warning', description='Save Threaded PDB', icon='save', style=ButtonStyle())

Output()

In [ ]:
download_button = widgets.Button(
    description="Download Threaded PDB",
    button_style="info",
    icon="download"
)

download_output = widgets.Output()


def download_threaded_pdb(_):

    with download_output:

        clear_output()

        filename = (
            output_name_widget.value.strip()
        )

        if not filename:
            filename = "threaded_model.pdb"

        if not filename.lower().endswith(
            ".pdb"
        ):
            filename += ".pdb"

        output_path = os.path.join(
            "/content",
            filename
        )

        if not os.path.exists(
            output_path
        ):
            print(
                "The threaded PDB has not been "
                "saved yet."
            )
            return

        files.download(
            output_path
        )


download_button.on_click(
    download_threaded_pdb
)

display(
    download_button,
    download_output
)

Button(button_style='info', description='Download Threaded PDB', icon='download', style=ButtonStyle())

Output()